# LLM Evaluation & LLM-as-a-Judge

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/building-with-llms/08-llm-evaluation

We simulate an LLM judge with position bias and show how order-swapping debiases pairwise evaluation — and why pairwise beats pointwise.

Self-contained: NumPy + matplotlib only. No torch, no sklearn, no network, no API keys.

> **To save your work:** click **Copy to Drive** at the top, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. A biased pairwise judge

Model A is genuinely better than B (true win prob 0.65). But our judge also has a **position bias**: it adds a fixed boost to whichever answer is shown *first*. If we always show A first, we overestimate A.

In [ ]:
def judge(true_p_A, first, position_boost=0.15, seed=0):
    """Return 'A' or 'B'. `first` is which answer is shown first."""
    g = np.random.default_rng(seed)
    p = true_p_A + (position_boost if first == 'A' else -position_boost)
    p = np.clip(p, 0, 1)
    return 'A' if g.random() < p else 'B'

# Always A-first: inflated estimate of A's win rate
N = 5000
wins_A_biased = np.mean([judge(0.65, 'A', seed=i) == 'A' for i in range(N)])
print('A win-rate, always A-first (biased):', round(wins_A_biased, 3), '(true 0.65)')

## 2. Debias by swapping order and averaging

Run each comparison twice — once with A first, once with B first — and average. The position boost cancels.

In [ ]:
def debiased_winrate(true_p_A, N=5000):
    a_first = np.mean([judge(true_p_A, 'A', seed=i) == 'A' for i in range(N)])
    b_first = np.mean([judge(true_p_A, 'B', seed=10_000+i) == 'A' for i in range(N)])
    return 0.5 * (a_first + b_first)

print('A win-rate, order-swapped (debiased):', round(debiased_winrate(0.65), 3), '(true 0.65)')

labels = ['always A-first', 'order-swapped', 'truth']
vals = [wins_A_biased, debiased_winrate(0.65), 0.65]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, [v*100 for v in vals], color=[ROSE, TEAL, BRAND])
ax.axhline(65, color=YELLOW, ls='--', lw=1, alpha=0.7)
ax.set_ylabel('estimated A win-rate (%)'); ax.set_title('Order-swapping cancels position bias')
ax.grid(True, alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

## 3. An eval suite score

An eval suite runs a metric over a fixed dataset and returns the mean — a single number you can gate releases on. Here a toy faithfulness check: does the answer only use facts present in the context?

In [ ]:
suite = [
    ({'context': {'paris', 'france'}, 'answer': {'paris'}}, ),
    ({'context': {'tokyo', 'japan'}, 'answer': {'tokyo', 'osaka'}}, ),  # 'osaka' not in context -> unfaithful
    ({'context': {'rome', 'italy'}, 'answer': {'rome'}}, ),
]
def faithful(ex):
    return ex['answer'].issubset(ex['context'])
score = np.mean([faithful(ex[0]) for ex in suite])
print('faithfulness suite score:', round(score, 3))

## 4. Reference-based metrics: perplexity and n-gram overlap

Sections 1–3 were all *reference-free*: a judge or an eval suite scoring a property of the output with no gold answer to compare against. When a gold reference **does** exist, two cheaper families of metric apply directly:

- **Cross-entropy / perplexity** — for a model that outputs a full probability distribution over classes/tokens, cross-entropy against the true (one-hot) label is $-\log p(\text{correct class})$, averaged over examples. Perplexity is just $\text{PPL} = e^{\text{cross-entropy}}$: a model that is certain *and* correct pushes this toward 1; a maximally-uncertain (uniform) prediction pushes it toward the number of classes.
- **N-gram overlap (BLEU/ROUGE-L)** — for free-form generated text, compare candidate tokens against a reference. The full from-scratch BLEU/ROUGE/METEOR derivation lives in <a href="https://ml-viz-ruby.vercel.app/wiki/text-generation-metrics">wiki/text-generation-metrics</a>; here's a short worked ROUGE-L (longest-common-subsequence F1) so it's not just a link-out.

In [ ]:
def lcs_length(a, b):
    """Length of the longest common subsequence of token lists a, b."""
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            dp[i][j] = dp[i - 1][j - 1] + 1 if a[i - 1] == b[j - 1] else max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]

def rouge_l(candidate, reference, beta=1.2):
    """ROUGE-L F-score: LCS-based precision/recall, order-sensitive but not contiguous."""
    l = lcs_length(candidate, reference)
    if l == 0:
        return 0.0
    p, r = l / len(candidate), l / len(reference)
    return ((1 + beta ** 2) * p * r) / (r + beta ** 2 * p)

reference = "the model correctly cited the source document".split()
candidate = "the model cited the correct source doc".split()
print('ROUGE-L F =', round(rouge_l(candidate, reference), 4))

## ✏️ Your turn — debiased pairwise win-rate

Implement `debiased(a_first_rate, b_first_rate)` = the average of the two orderings' A-win-rates.

In [ ]:
def debiased(a_first_rate, b_first_rate):
    """TODO(you): return the mean of the two rates."""
    # TODO
    return ...


In [ ]:
assert abs(debiased(0.80, 0.50) - 0.65) < 1e-9
assert abs(debiased(0.60, 0.60) - 0.60) < 1e-9
print('✅ averaging both orders cancels the position boost.')

<details>
<summary>Solution</summary>

```python
def debiased(a_first_rate, b_first_rate):
    return 0.5 * (a_first_rate + b_first_rate)
```

Prefer **pairwise** over pointwise (models rank better than they grade), swap order to kill position bias, control for length, and validate the judge against human labels before trusting it.
</details>

## 🔬 Extra practice — DML `134_compute-multi-class-cross-entropy-loss`

<a href="https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/134_compute-multi-class-cross-entropy-loss">Open-Deep-ML problem 134</a> asks for the average cross-entropy loss over a batch of predicted class-probability rows and one-hot true-label rows, clipping probabilities by `epsilon` for numerical stability so `log(0)` never appears. Match its signature:

```python
def compute_cross_entropy_loss(predicted_probs: np.ndarray, true_labels: np.ndarray, epsilon=1e-15) -> float:
    ...
```

This is exactly the loss whose exponential *is* the perplexity from the section above. Two edge cases worth building intuition for: a **perfect** prediction (probability 1 on every true class) should cost (essentially) zero, while a **uniform** prediction that puts no information anywhere should always cost exactly $\log(C)$ for $C$ classes — no matter which class happens to be true.

In [ ]:
def compute_cross_entropy_loss(predicted_probs, true_labels, epsilon=1e-15):
    """DML 134: average cross-entropy loss over a batch of one-hot-labeled rows."""
    predicted_probs = np.asarray(predicted_probs, dtype=float)
    true_labels = np.asarray(true_labels, dtype=float)

    # TODO(you): clip predicted_probs into [epsilon, 1 - epsilon] so log(0) never happens
    # (hint: np.clip)
    clipped = ...

    # TODO(you): per-row loss is -sum(true * log(pred)) over classes, then average over rows
    # (hint: np.log, np.sum(..., axis=1), .mean())
    return ...

In [ ]:
# Checks -- run me (DML's own published test cases, from tests.json, plus edge cases)
pred1 = np.array([[1, 0, 0], [0, 1, 0]])
true1 = np.array([[1, 0, 0], [0, 1, 0]])
assert abs(compute_cross_entropy_loss(pred1, true1)) < 1e-9, "perfect predictions -> (near) zero cross-entropy"

pred2 = np.array([[0.1, 0.8, 0.1], [0.8, 0.1, 0.1]])
true2 = np.array([[0, 0, 1], [0, 1, 0]])
assert abs(compute_cross_entropy_loss(pred2, true2) - 2.3026) < 1e-4

pred3 = np.array([[0.7, 0.2, 0.1], [0.3, 0.6, 0.1]])
true3 = np.array([[1, 0, 0], [0, 1, 0]])
assert abs(compute_cross_entropy_loss(pred3, true3) - 0.4338) < 1e-4

# Edge case: uniform prediction distribution over C classes -- with zero
# information, the loss must equal exactly log(C), regardless of the true label.
C = 4
uniform_pred = np.full((3, C), 1.0 / C)
true_uniform = np.eye(C)[[0, 2, 3]]  # three distinct one-hot rows
assert abs(compute_cross_entropy_loss(uniform_pred, true_uniform) - np.log(C)) < 1e-9, \
    "uniform prediction over C classes always costs log(C)"

print('✅ perfect predictions -> ~0, uniform predictions -> log(C), and DML\'s published cases all match.')

<details><summary>💡 Show solution</summary>

```python
def compute_cross_entropy_loss(predicted_probs, true_labels, epsilon=1e-15):
    predicted_probs = np.asarray(predicted_probs, dtype=float)
    true_labels = np.asarray(true_labels, dtype=float)
    clipped = np.clip(predicted_probs, epsilon, 1 - epsilon)
    return float(np.mean(-np.sum(true_labels * np.log(clipped), axis=1)))
```

Clipping keeps `log(0)` from appearing when a model assigns exactly probability 0 to the true class — without it, a single overconfident wrong prediction would blow the whole batch's average up to infinity. Note `np.exp(compute_cross_entropy_loss(...))` recovers exactly the perplexity from the section above.
</details>

## Recap

- LLM judges have **position, verbosity, and self-preference** biases — design around them.
- **Order-swapping and averaging** cancels position bias in pairwise evaluation.
- An **eval suite** = dataset + metric → one gateable number; run it offline (release gate) and online (sampled traffic).
- Grow the suite from real production failures so it sharpens over time.